# 01. Apollo Baseline

This notebook runs the full cross-validation benchmark on the purified tumor data.
It implements the exact methodology from the Apollo paper: Feature Selection (Top 20) + SVM.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
import joblib

# Add project root to path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.models.baselines import ApolloPipeline

In [2]:
# --- Configuration ---
DATA_PATH = "../data/processed/glioma_dataset.h5"
SPLIT_FILE = "../data/splits/manual_splits.json"

if not os.path.exists(SPLIT_FILE):
    print(f"❌ ERROR: Splits file not found at {SPLIT_FILE}")

print("Initializing Loader...")
# Ensure tumor_only=True so we only train on the data identified by our Spatio-Spectral Annotator
loader = GliomaDataLoader(
    data_path=DATA_PATH, 
    split_file=SPLIT_FILE, 
    memory_mapped=False, 
    tumor_only=True
)

print(f"✅ Loader initialized. Total Tumor Spectra: {loader.X.shape[0]}")

Initializing Loader...
📂 Loading data from ..\data\processed\glioma_dataset.h5...
🔬 'tumor_only=True': Filtering out healthy tissue...
   ✅ Filtered to 88487 Pure Tumor Spectra.
✅ Loader initialized. Total Tumor Spectra: 88487


## Run Benchmark (5-Fold CV)

In [ ]:
# --- 5-Fold Grouped Cross-Validation ---
# We iterate directly over the loader, which handles the splitting logic internally.

fold_results = []
feature_masks = []

print(f"🚀 Starting Training via Loader Iterator...")

for fold_data in loader:
    fold_id = fold_data['fold_name']
    print(f"\n>>> Running {fold_id} <<<")
    
    # Unpack Data
    X_train, y_train = fold_data['train']
    X_val, y_val = fold_data['val']
    
    print(f"   Train Shape: {X_train.shape}, Val Shape: {X_val.shape}")
    
    # 2. Instantiate Apollo Pipeline 
    # (Re-instantiate per fold to ensure a fresh start)
    model = ApolloPipeline(n_features=20, random_state=42)
    
    # 3. Train
    # This runs: RF Selector -> Top 20 Features -> Scaler -> SVM
    model.fit(X_train, y_train)
    
    # Save the selected features (Critical for Phase 3: TSTR)
    feature_masks.append(model.selected_indices_)
    
    # 5. Evaluate
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    print(f"    > Fold Accuracy: {acc:.4f}")

    os.makedirs("../models/baselines", exist_ok=True)
    model_path = f"../models/baselines/apollo_{fold_id}.joblib"
    joblib.dump(model, model_path)
    print(f"    > Model saved to {model_path}")
    
    # 6. Track Results
    fold_results.append({
        'fold': fold_idx + 1,
        'accuracy': acc,
        'y_true': y_val,
        'y_pred': y_pred
    })
    
    # 7. Track Selected Features
    feature_masks.append(model.selected_indices_)
    
    print(f"   ✅ Fold Accuracy: {acc:.4f}")

# --- Summary ---
mean_acc = np.mean([r['accuracy'] for r in fold_results])
print(f"\n🏆 Average Accuracy: {mean_acc:.4f}")

## Analysis & Reporting

In [ ]:
# Aggregate Accuracy
accuracies = [r['accuracy'] for r in fold_results]
mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)

print(f"\n=== Final Results ===")
print(f"Apollo Baseline Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")

# Create results directory if needed
os.makedirs("../results", exist_ok=True)

# Save simple metrics
metrics_df = pd.DataFrame(fold_results).drop(columns=['y_true', 'y_pred'])
metrics_df.to_csv("../results/baseline_metrics.csv", index=False)
print("✅ Saved clean metrics CSV.")

# Save feature masks for later TSTR evaluation
np.save("../results/feature_masks.npy", feature_masks)
print("✅ Saved selected feature masks to .npy")

In [ ]:
all_true = np.concatenate([r['y_true'] for r in fold_results])
all_pred = np.concatenate([r['y_pred'] for r in fold_results])

cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f"Aggregated Confusion Matrix (Acc: {mean_acc:.4f})")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.show()

In [ ]:
# # Future Usage: To make predictions on new spectra, simply load the trained model and call predict. The pipeline will handle feature selection and scaling automatically.
# loaded_model = joblib.load("../models/baselines/apollo_fold_1.joblib")
# prediction = loaded_model.predict(new_spectrum) 
# # ^ This handles feature selection and scaling automatically!

In [10]:
import time
import os
import gc
import joblib
import numpy as np
import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from src.models.cnn import build_1d_cnn 

# --- Setup Output Directories ---
MODEL_DIR = "../results/models/baselines"
os.makedirs(MODEL_DIR, exist_ok=True)

benchmark_results = {
    'Apollo': [],
    'RF_Standard': [],
    'XGBoost': [],
    '1D_CNN': []
}

print("🚀 Starting Memory-Optimized Multi-Model TRTR Benchmark...")

for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"\n{'='*40}")
    print(f"🔄 Processing {fold_name}")
    print(f"{'='*40}")
    
    # 1. Unpack 
    X_train = fold_data['train'][0]
    y_train = fold_data['train'][1]
    X_val = fold_data['val'][0]
    y_val = fold_data['val'][1]
    
    # Fix 1-indexed labels if necessary
    if np.min(y_train) == 1:
        y_train = y_train - 1
        y_val = y_val - 1
        
    # --- MODEL 1: Apollo Baseline ---
    print("Training Apollo Pipeline...")
    # (Assuming apollo_pipeline is defined in a cell above this loop)
    apollo_pipeline.fit(X_train, y_train)
    y_pred_apollo = apollo_pipeline.predict(X_val)
    benchmark_results['Apollo'].append(accuracy_score(y_val, y_pred_apollo))
    
    # 💾 Save and Flush
    joblib.dump(apollo_pipeline, f"{MODEL_DIR}/apollo_{fold_name}.joblib")
    del y_pred_apollo # Keep pipeline in memory if defined globally, but delete preds
    gc.collect()
    
    # --- MODEL 2: Standard Random Forest ---
    print("Training Standard Random Forest...")
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_val)
    benchmark_results['RF_Standard'].append(accuracy_score(y_val, y_pred_rf))
    
    # 💾 Save and Flush
    joblib.dump(rf, f"{MODEL_DIR}/rf_{fold_name}.joblib")
    del rf, y_pred_rf
    gc.collect()
    
    # --- MODEL 3: XGBoost ---
    print("Training XGBoost...")
    # tree_method='hist' is faster and more memory efficient for large datasets
    xgb = XGBClassifier(objective='multi:softprob', num_class=6, random_state=42, n_jobs=-1, tree_method='hist')
    xgb.fit(X_train, y_train)
    y_pred_xgb = xgb.predict(X_val)
    benchmark_results['XGBoost'].append(accuracy_score(y_val, y_pred_xgb))
    
    # 💾 Save and Flush (XGBoost prefers saving as .json instead of joblib)
    xgb.save_model(f"{MODEL_DIR}/xgb_{fold_name}.json")
    del xgb, y_pred_xgb
    gc.collect()
    
    # --- MODEL 4: 1D-CNN ---
    print("Training 1D-CNN...")
    tf.keras.backend.clear_session()
    
    X_train_cnn = X_train.reshape(-1, 1738, 1)
    X_val_cnn = X_val.reshape(-1, 1738, 1)
    y_train_cnn = tf.keras.utils.to_categorical(y_train, num_classes=6)
    y_val_cnn = tf.keras.utils.to_categorical(y_val, num_classes=6)
    
    cnn_model = build_1d_cnn(input_shape=(1738, 1), num_classes=6)
    cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    
    cnn_model.fit(X_train_cnn, y_train_cnn, 
                  validation_data=(X_val_cnn, y_val_cnn),
                  epochs=30, batch_size=256, 
                  callbacks=[early_stop], verbose=0)
                  
    y_pred_cnn_probs = cnn_model.predict(X_val_cnn, verbose=0)
    y_pred_cnn = np.argmax(y_pred_cnn_probs, axis=1)
    benchmark_results['1D_CNN'].append(accuracy_score(y_val, y_pred_cnn))
    
    # 💾 Save and Flush
    cnn_model.save(f"{MODEL_DIR}/cnn_{fold_name}.keras")
    del cnn_model, X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn, y_pred_cnn_probs, y_pred_cnn
    tf.keras.backend.clear_session()
    gc.collect()

# --- Print Final Summary ---
print("\n🎉 === TRTR BENCHMARK COMPLETE ===")
for model_name, accuracies in benchmark_results.items():
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    print(f"{model_name:>15}: {mean_acc:.4f} ± {std_acc:.4f}")

🚀 Starting Memory-Optimized Multi-Model TRTR Benchmark...

🔄 Processing fold_0
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (66253, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Training Standard Random Forest...
Training XGBoost...
Training 1D-CNN...

🔄 Processing fold_1
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (68177, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Training Standard Random Forest...
Training XGBoost...
Training 1D-CNN...

🔄 Processing fold_2
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (66791, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Training Standard Random Forest...
Training XGBoost...
Training 1D-CNN...

🔄 Processing fold_3
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (75116, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Trai

In [ ]:
import numpy as np
import joblib
import xgboost as xgb
import tensorflow as tf
from sklearn.metrics import accuracy_score
import gc

MODEL_DIR = "../results/models/baselines"

six_class_benchmark_results = {
    'Apollo': [],
    'RF_Standard': [],
    'XGBoost': [],
    '1D_CNN': []
}

binary_benchmark_results = {
    'Apollo (Mapped to Binary)': [],
    'RF_Standard (Mapped to Binary)': [],
    'XGBoost (Mapped to Binary)': [],
    '1D_CNN (Mapped to Binary)': []
}

print("🔍 Testing 6-Class Models on the Binary Task...")

for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"🔄 Evaluating {fold_name}...")
    
    # Load validation data
    X_val = fold_data['val'][0].astype(np.float32)
    y_val_6_class = fold_data['val'][1]
    
    # Fix 1-indexed to 0-indexed if necessary
    if np.min(y_val_6_class) == 1:
        y_val_6_class = y_val_6_class - 1
        
    # Map True Labels to Binary: 0,1,2 -> 0 (Mutant) | 3,4,5 -> 1 (Wildtype)
    y_val_binary = np.where(y_val_6_class < 3, 0, 1)
    
    # --- 1. Apollo ---
    apollo = joblib.load(f"{MODEL_DIR}/apollo_{fold_name}.joblib")
    y_pred_apollo_6 = apollo.predict(X_val)
    y_pred_apollo_bin = np.where(y_pred_apollo_6 < 3, 0, 1)
    binary_benchmark_results['Apollo (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_apollo_bin))
    
    # --- 2. Random Forest ---
    rf = joblib.load(f"{MODEL_DIR}/rf_{fold_name}.joblib")
    y_pred_rf_6 = rf.predict(X_val)
    y_pred_rf_bin = np.where(y_pred_rf_6 < 3, 0, 1)
    binary_benchmark_results['RF_Standard (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_rf_bin))
    
    # --- 3. XGBoost ---
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model(f"{MODEL_DIR}/xgb_{fold_name}.json")
    y_pred_xgb_6 = xgb_model.predict(X_val)
    y_pred_xgb_bin = np.where(y_pred_xgb_6 < 3, 0, 1)
    binary_benchmark_results['XGBoost (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_xgb_bin))
    
    # --- 4. 1D-CNN ---
    cnn = tf.keras.models.load_model(f"{MODEL_DIR}/cnn_{fold_name}.keras")
    X_val_cnn = X_val.reshape(-1, 1738, 1)
    y_pred_cnn_probs = cnn.predict(X_val_cnn, verbose=0)
    y_pred_cnn_6 = np.argmax(y_pred_cnn_probs, axis=1)
    y_pred_cnn_bin = np.where(y_pred_cnn_6 < 3, 0, 1)
    binary_benchmark_results['1D_CNN (Mapped to Binary)'].append(accuracy_score(y_val_binary, y_pred_cnn_bin))
    
    # Cleanup memory
    del apollo, rf, xgb_model, cnn
    tf.keras.backend.clear_session()
    gc.collect()

print("\n🎉 === BINARY BENCHMARK (MAPPED FROM 6-CLASS) ===")
for model_name, accuracies in binary_benchmark_results.items():
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    print(f"{model_name:>35}: {mean_acc:.4f} ± {std_acc:.4f}")

🔍 Testing 6-Class Models on the Binary Task...
🔄 Evaluating fold_0...

🔄 Evaluating fold_1...
🔄 Evaluating fold_2...
🔄 Evaluating fold_3...
🔄 Evaluating fold_4...

🎉 === BINARY BENCHMARK (MAPPED FROM 6-CLASS) ===
          Apollo (Mapped to Binary): 0.6666 ± 0.0629
     RF_Standard (Mapped to Binary): 0.6604 ± 0.1199
         XGBoost (Mapped to Binary): 0.7135 ± 0.1262
          1D_CNN (Mapped to Binary): 0.4384 ± 0.0973
